In [0]:
%run ./utils

In [0]:
from datetime import datetime, timedelta, timezone
from functools import reduce
import pyspark.sql.functions as F

def get_checksum_df(start_time, end_time, config_id_list, landing_consumer_path, config_database):
    """
    Return (loading_detail_df, cbr_detail_df) for the given UTC time window.
    Both DataFrames have schema: Market, Topic, H00~H23, DailyTotal.
    config_id_list: list of downstream_config.config_id values to report on.
    landing_consumer_path: Delta path for bronze landing consumer data.
    config_database: database name containing downstream_log/downstream_config.
    Either landing_consumer_path or config_id_list/config_database can be empty/None to skip that section.
    """

    # ---------- Loading consumers (bronze landing) ----------
    if landing_consumer_path and landing_consumer_path.strip():
        loading_hours = [f"H{h:02d}" for h in range(24)]
        loading_detail = (
            spark.read.format("delta").load(landing_consumer_path)
            .filter(
                (F.col("slndc_creation_dt") >= F.lit(start_time)) &
                (F.col("slndc_creation_dt") <  F.lit(end_time))
            )
            .withColumn(
                "Market",
                F.coalesce(
                    F.get_json_object(F.col("slndc_payload"), "$.source.Consumer.SourceSystem.MarketCode"),
                    F.lit("UNKNOWN")
                )
            )
            .withColumnRenamed("slndc_kafka_topic", "Topic")
            .withColumn("Hour", F.concat(F.lit("H"), F.lpad(F.hour("slndc_creation_dt").cast("string"), 2, "0")))
            .groupBy("Market", "Topic")
            .pivot("Hour", loading_hours)
            .agg(F.count("slndc_id"))
            .fillna(0, subset=loading_hours)
            .withColumn("DailyTotal", reduce(lambda a, b: a + b, [F.col(h) for h in loading_hours]))
            .fillna("UNKNOWN", subset=["Market", "Topic"])
            .orderBy(F.col("DailyTotal").desc(), F.col("Market").asc(), F.col("Topic").asc())
        )
    else:
        loading_detail = None

    # ---------- CBR downstream (downstream_log -> downstream_config) ----------
    if config_id_list and config_database and config_database.strip():
        log_tbl = f"{config_database}.downstream_log"
        config_tbl = f"{config_database}.downstream_config"

        log_df = (
            spark.table(log_tbl)
            .filter(
                (F.col("create_time") >= F.lit(start_time)) &
                (F.col("create_time") <  F.lit(end_time))
            )
        )

        config_df = (
            spark.table(config_tbl)
            .filter(F.col("is_active") == True)
            .filter(F.col("config_id").isin(config_id_list))
            .select("config_id", "marketcode", "kafka_topic_name")
        )

        cbr_hours = [f"H{h:02d}" for h in range(24)]
        cbr_detail = (
            log_df
            .join(config_df, on="config_id", how="inner")
            .withColumnRenamed("marketcode", "Market")
            .withColumnRenamed("kafka_topic_name", "Topic")
            .withColumn("Hour", F.concat(F.lit("H"), F.lpad(F.hour("create_time").cast("string"), 2, "0")))
            .groupBy("Market", "Topic")
            .pivot("Hour", cbr_hours)
            .agg(F.coalesce(F.sum("downstream_row_count"), F.lit(0)))
            .fillna(0, subset=cbr_hours)
            .withColumn("DailyTotal", reduce(lambda a, b: a + b, [F.col(h) for h in cbr_hours]))
            .fillna("UNKNOWN", subset=["Market", "Topic"])
            .orderBy(F.col("DailyTotal").desc(), F.col("Market").asc(), F.col("Topic").asc())
        )
    else:
        cbr_detail = None

    return loading_detail, cbr_detail


In [0]:
def build_email_body(report_date, start_time, end_time, monitor_id, loading_html, cbr_html):
    """Build the full HTML email body for the daily volume report."""
    return f"""
    <html>
      <head>
        <style>
          .data-table {{
            border-collapse: collapse;
            width: 70%;
            font-family: Arial, sans-serif;
            font-size: 12px;
          }}
          .data-table th, .data-table td {{
            border: 1px solid #ccc;
            padding: 6px;
            text-align: left;
          }}
          .data-table th {{
            background-color: #f8f9fa;
            border-bottom: 2px solid #666;
          }}
        </style>
      </head>
      <body>
        <h2>Daily Volume Report for {report_date.isoformat()} (UTC)</h2>
        <p>Check time period(UTC): {start_time} -> {end_time}. <br>monitor_id: {monitor_id}</p>

        <h3>1. Loading Consumers (bronze landing)</h3>
        {loading_html}

        <h3>2. CBR Downstream (downstream_log)</h3>
        {cbr_html}

      </body>
    </html>
    """


In [0]:
def monitor_main(monitor_id, start_time, end_time, config_id_list, landing_consumer_path, config_database, max_rows=MAX_ROWS, max_cols=MAX_COLS, to_addrs=None):
    loading_df, cbr_df = get_checksum_df(start_time, end_time, config_id_list, landing_consumer_path, config_database)

    if loading_df is not None:
        loading_df.cache()
    if cbr_df is not None:
        cbr_df.cache()

    try:
        print(f"Daily volume summary: {monitor_id}")
        if loading_df is not None:
            print("Displaying Loading Consumers (bronze landing) summary:")
            display(loading_df)
        if cbr_df is not None:
            print("Displaying CBR Downstream (downstream_log) summary:")
            display(cbr_df)

        report_date = start_time.date()
        loading_html, _, _ = build_html_table_fragment(loading_df, max_rows=max_rows, max_cols=max_cols) if loading_df is not None else ("<p>Landing consumer path not configured; skipping landing volume.</p>", "0", "0")
        cbr_html, _, _ = build_html_table_fragment(cbr_df, max_rows=max_rows, max_cols=max_cols) if cbr_df is not None else ("<p>Config id list not provided; skipping CBR volume.</p>", "0", "0")

        html_body = build_email_body(report_date, start_time, end_time, monitor_id, loading_html, cbr_html)

        recipients = to_addrs or TO_ADDRS
        if not recipients:
            raise ValueError("to_addrs is empty; no recipients configured for the daily volume report email.")

        send_email(
            subject=SUBJECT.format(yyyymmdd=end_time.strftime("%Y%m%d")),
            html_body=html_body,
            to_addrs=recipients,
            cc_addrs=CC_ADDRS,
            bcc_addrs=BCC_ADDRS,
            custom_text=""
        )

        print(f"Daily volume email sent for {report_date.isoformat()}: {monitor_id}")

    finally:
        if loading_df is not None:
            loading_df.unpersist()
        if cbr_df is not None:
            cbr_df.unpersist()


In [0]:
TO_ADDRS: List[str] = []
CC_ADDRS: List[str] = []
BCC_ADDRS: List[str] = []

SUBJECT = "[Checksum] [MDM] CBR Checksum {yyyymmdd}"

In [0]:
monitor_id = dbutils.widgets.get("monitor_id")

# Landing consumer Delta path. None or empty to skip loading-consumer statistics.
landing_consumer_path = dbutils.widgets.get("landing_consumer_path") or None

# Database containing downstream_log and downstream_config. None or empty to skip CBR statistics.
config_database = dbutils.widgets.get("config_database") or None

# Comma-separated list of downstream_config.config_id values, matching downstream_run.py.
config_id_list_str = dbutils.widgets.get("config_id_list")
config_id_list = [x.strip() for x in config_id_list_str.split(",") if x.strip()] if config_id_list_str else []

try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except:
    trigger_timestamp_ms = int(datetime.now().timestamp())

# Maximum rows/columns to show in the email HTML tables.
try:
    max_rows = int(dbutils.widgets.get("max_rows"))
except:
    max_rows = MAX_ROWS

try:
    max_cols = int(dbutils.widgets.get("max_cols"))
except:
    max_cols = MAX_COLS

# Comma-separated list of recipient email addresses.
to_addrs_str = dbutils.widgets.get("to_addrs")
to_addrs = [x.strip() for x in to_addrs_str.split(",") if x.strip()] if to_addrs_str else TO_ADDRS

# This monitor always reports the full day *before* the trigger timestamp.
trigger_time = datetime.fromtimestamp(trigger_timestamp_ms, tz=timezone.utc)
report_date = (trigger_time - timedelta(days=1)).date()
start_time = datetime(report_date.year, report_date.month, report_date.day, tzinfo=timezone.utc)
end_time = start_time + timedelta(days=1)

print(f"monitor_id: {monitor_id}")
print(f"landing_consumer_path: {landing_consumer_path}")
print(f"config_database: {config_database}")
print(f"config_id_list: {config_id_list}")
print(f"max_rows: {max_rows}, max_cols: {max_cols}")
print(f"to_addrs: {to_addrs}")
print(f"report_date: {report_date}, start_time: {start_time}, end_time: {end_time}")

monitor_main(monitor_id, start_time, end_time, config_id_list, landing_consumer_path, config_database, max_rows, max_cols, to_addrs)
